# Principal Component Analysis (PCA)

> **Core Idea:** Find new axes in the data that capture maximum variance, then project data onto fewer of those axes — reducing dimensions while preserving as much information as possible.

---

| Section | Topic |
|---------|-------|
| 1 | What is PCA & Why it Exists |
| 2 | Geometric Intuition |
| 3 | Why Variance is the Key |
| 4 | What are Principal Components? |

## 1. What is PCA & Why Does It Exist?

### The Problem: Datasets Are High-Dimensional

Every column in your dataset is a dimension (feature). Real-world data grows very fast:

| Data Type | Dimensions |
|-----------|-----------|
| MNIST digit image (28×28 grayscale) | 784 |
| Color photo (256×256 RGB) | 196,608 |
| Text — bag-of-words representation | 3,000 – 50,000 |
| Gene expression data | 20,000+ |

This creates three compounding problems:

| Problem | Effect |
|---------|--------|
| **Slow training** | Every extra feature adds parameters and computation |
| **Overfitting** | Model learns noise in irrelevant dimensions |
| **No visualization** | You cannot plot data beyond 3D |

---

### The Photography Analogy

A photographer at a 3D football match projects the entire 3D scene onto a flat 2D sensor. Depth is partially lost, but the essential structure — shapes, positions, patterns — is preserved in the photo.

```
3D scene   ──[ camera ]──>  2D photograph   (some depth lost, structure preserved)
784-D MNIST ──[  PCA  ]──>  2D plot         (some variance lost, patterns preserved)
10-D data   ──[  PCA  ]──>  3D / 2D space
```

PCA is that camera. It finds the **best projection angle** — the one that preserves the most spread in the data.

---

### Benefits

| Benefit | Description |
|---------|-------------|
| **Faster algorithm execution** | Fewer features → fewer computations |
| **Visualization** | Compress 784-D or 10-D data to 2D/3D for plotting |
| **Noise reduction** | Low-variance dimensions often encode noise, not signal |
| **Decorrelation** | PCA components are always orthogonal (zero correlation) |

## 2. Geometric Intuition

### Example: Student Exam Scores

Suppose you have scores for 6 students across 3 subjects:

| Student | Math | Physics | Chemistry |
|---------|------|---------|-----------|
| A | 88 | 85 | 82 |
| B | 72 | 70 | 68 |
| C | 94 | 91 | 89 |
| D | 60 | 58 | 62 |
| E | 80 | 77 | 75 |
| F | 50 | 53 | 55 |

**Observation:** Math, Physics, and Chemistry scores move almost together. A student good at Math is likely good at Physics too.  
These 3 features are **highly correlated** — they're basically saying the same thing: *"how academically strong is this student?"*

> PCA detects this redundancy and compresses 3 correlated features into 1 principal component that captures the shared "academic ability" direction.

---

### What Projection Means

Consider just Math (x) vs Physics (y). The data lies along a diagonal:

```
Physics
   |           C
   |        E     A
   |     B
   |  F       D
   +───────────────── Math
```

The data is elongated diagonally — most variance runs along that diagonal, not along either axis.

**Feature Selection approach:** Drop Physics, keep only Math.
→ You project every point straight down onto the x-axis.
→ Distance between points along that axis: `d'` (compressed — some separation is lost)

**PCA approach:** Rotate the axes so the new x-axis *aligns with the diagonal*.
→ Project onto this better axis.
→ Distance between points: `d ≈ d'` (nearly the same as original — very little info lost)

```
Feature Selection:   d  >  d'    (information lost — projecting onto wrong axis)
Feature Extraction:  d  ≈  d'    (nearly lossless — projecting onto best axis)
```

---

### The Key Distinction: PCA vs Feature Selection

| | Feature Selection | Feature Extraction (PCA) |
|--|------------------|-----------------------|
| Method | Keep original columns, drop others | Create new columns as linear combinations |
| Example | Keep "Math", drop "Physics", drop "Chemistry" | Create "PC1 = 0.58·Math + 0.58·Physics + 0.57·Chemistry" |
| Interpretability | High — original feature names preserved | Lower — components are abstract |
| Information preserved | Moderate | Maximum possible for given `k` |

> PCA does not select existing columns. It **creates new axes** that are tilted to align with the natural directions of spread in the data.

## 3. Why Variance is the Key

### What is Variance?

Variance quantifies **spread** — how far data points are from their mean.

$$\text{Variance} = \frac{\sum_{i=1}^{n}(x_i - \bar{x})^2}{n}$$

---

### Intuition: Same Mean, Different Spread

Consider two sets of exam scores, both with mean = 70:

```
Set A:   ──●──────●──────●──
          65      70     75        Variance = 25/3 ≈ 8.3   (tight cluster)

Set B:   ──●──────────────●──────────────●──
          50              70             90        Variance = 400/3 ≈ 133  (widely spread)
```

Both have the same average. But Set B has far more **discriminating power** — you can tell students apart.

> If variance → 0, all points collapse to a single location. Projecting onto that axis gives you nothing useful.  
> **Maximum variance = maximum information = best axis to project onto.**

---

### Connecting Variance to PCA's Objective

When you project data onto an axis, the goal is to **maximise the spread of the projections**:

```
Bad axis (low variance after projection):

   ×   ×  ×   ×  ×   ×
─────────────────────────────▶  axis
      ● ● ● ● ●               (all projections squished — can't distinguish points)


Good axis (high variance after projection):

   ×         ×      ×         ×
──────────────────────────────▶  axis
   ●         ●      ●         ●  (projections spread out — points are distinguishable)
```

**PC1** = the axis that maximises the variance of the projected data.  
**PC2** = the next best orthogonal axis.  
**PC3, PC4, ...** = remaining orthogonal axes, each explaining less variance than the previous.

---

### Visualisation — Projection onto Two Candidate Axes

The cell below plots the student score data and shows why the diagonal direction (PC1) preserves more separation than projecting onto the original Math axis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Student exam scores (Math, Physics)
np.random.seed(42)
math    = np.array([88, 72, 94, 60, 80, 50], dtype=float)
physics = np.array([85, 70, 91, 58, 77, 53], dtype=float)
labels  = ['A', 'B', 'C', 'D', 'E', 'F']

X = np.stack([math, physics], axis=1)
X_centered = X - X.mean(axis=0)

# Compute PCA direction (eigenvector of cov matrix)
cov = np.cov(X_centered.T)
eigvals, eigvecs = np.linalg.eigh(cov)
pc1 = eigvecs[:, np.argmax(eigvals)]  # direction of max variance

# Project onto PC1 and onto original Math axis
proj_pc1   = np.outer(X_centered @ pc1, pc1)          # onto diagonal
proj_math  = X_centered.copy(); proj_math[:, 1] = 0   # onto Math axis (x-axis)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Projection: Original Axis vs PCA Axis", fontsize=14, fontweight='bold', y=1.02)

for ax, proj, title, color in zip(
    axes,
    [proj_math, proj_pc1],
    ["Feature Selection\n(project onto Math axis)", "PCA\n(project onto PC1 — diagonal)"],
    ["#e74c3c", "#2ecc71"]
):
    ax.scatter(*X_centered.T, color='#2c3e50', zorder=5, s=80)
    for i, lbl in enumerate(labels):
        ax.annotate(lbl, X_centered[i] + [0.4, 0.4], fontsize=9)

    # Draw projection lines
    for i in range(len(X_centered)):
        ax.plot([X_centered[i, 0], proj[i, 0]],
                [X_centered[i, 1], proj[i, 1]],
                color='grey', lw=0.8, linestyle='--', alpha=0.6)

    ax.scatter(*proj.T, color=color, zorder=4, s=60, marker='x', linewidths=2)

    # Draw the projection axis
    span = np.linspace(-20, 20, 100)
    if "PCA" in title:
        ax.plot(span * pc1[0], span * pc1[1], color=color, lw=1.5, alpha=0.5)
    else:
        ax.axhline(0, color=color, lw=1.5, alpha=0.5)

    # Variance of projected points
    var = np.var(proj @ pc1 if "PCA" in title else proj[:, 0])
    ax.set_title(f"{title}\nVariance of projections = {var:.1f}", fontsize=11)
    ax.set_xlabel("Math (centered)"); ax.set_ylabel("Physics (centered)")
    ax.set_aspect('equal')
    ax.axhline(0, color='black', lw=0.5); ax.axvline(0, color='black', lw=0.5)
    ax.grid(True, alpha=0.3)

    orig_patch = mpatches.Patch(color='#2c3e50', label='Original points')
    proj_patch = mpatches.Patch(color=color, label='Projected points')
    ax.legend(handles=[orig_patch, proj_patch], fontsize=8)

plt.tight_layout()
plt.show()
print("Higher variance on the projection axis → more information retained.")